In [ ]:
# using GNNs to build network graph from nmap output

# Read the XML `nmap` Output
- Parse `.xml` into nodes, edges, attributes.

https://www.youtube.com/watch?v=5SlemSWGD1g

Note:
What I have now is not universal, I will need our true `nmap` scans to make this one-size fits all.

In [ ]:
import xml.etree.ElementTree as ET  # ElementTree library is used to parse XML data

xml1 = "data/nmap_output.xml"
xml2 = "data/nmap_output_adv.xml"

tree = ET.parse(xml2)
root = tree.getroot()  # tag that envelopes everything
root.attrib  # length of 6 (scaninfo, host1, host2, host3, host4, runstats)

In [ ]:
# loop over root children and their sub attributes
# find each HOST element
network = {}
for child in root: 
    network_config = {}

    # skip over none host elements
    if child.tag != "host":
        continue

    # print("CHILD:", child)
    # pull all IP hosts found (up/down)
    addr = child.find("address").attrib["addr"]  # might not be universal
    status = child.find("status").attrib["state"]

    if status != "up":
        network_config["os"] = None
        network_config["state"] = None
        network_config["hostname"] = None
        network_config["ports"] = None
    else: 
        # find IP hostname
        hostname_root = child.find("hostnames")
        hostname = hostname_root.find("hostname").attrib["name"]

        # find IP OS
        os_config = {}
        os_root = child.find("os")
        if os_root is not None:
            os = os_root.find("osmatch")
            network_config["os"] = os.attrib
        else: 
            network_config["os"] = None

        network_config["state"] = status 
        network_config["hostname"] = hostname

        # find IP open ports  (might need to clean this up to make it more universal)
        port_lst = []
        port_root = child.find("ports")
        port_info = port_root.find("port")
        for port in port_root:
            for val in port:
                port.attrib |= val.attrib
            port_lst.append(port.attrib)

        network_config["ports"] = port_lst

    # add the host into the dictionary
    network[addr] = network_config
    
print(network)


## Turn ElementTree Dictionary into `networkx` graph.

In [ ]:
# print(network["192.168.1.2"]["state"])
print(network.keys())

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

G = nx.Graph()
SCANNER = "AGENT"
# parse the dictionary to get the nodes (scanner -> IP -> Ports)
G.add_node(SCANNER)

for ip in network.keys():
    G.add_node(ip, color="#4c956c" if network[ip]["state"] == "up" else "#d9d9d9")  # adding a node to IP

    # if down connect using dashed lines
    if network[ip]["state"] != "up":
        G.add_edge(SCANNER, ip)
    else:
        G.add_edge(SCANNER, ip)

    # add port edges
    if network[ip]["ports"] is not None:
        for n in network[ip]["ports"]:
            # color code port edges
            if n["portid"] in [80, 443]:
                color = "#fb8500"
            elif n["portid"] == 22:
                color = "#d9d9d9"
            else:
                color = "#0077b6"

            # print(n)
            service_label = f"{n['portid']}/{n.get('name', 'unknown')}"
            G.add_node(service_label)
            G.add_edge(ip, service_label, color=color)


# display graph
# pull the colors used
node_colors = [G.nodes[n].get("color", "#0077b6") for n in G.nodes()]
edge_colors = nx.get_edge_attributes(G, "color").values()
node_degree = G.degree
nx.draw(
    G,
    # pos=nx.multipartite_layout(G), 
    node_color=node_colors,
    edge_color=edge_colors,
    with_labels= True,
    node_size=[v[1] * 200 for v in node_degree]
)

plt.show()
plt.savefig("nmap_out_adv.png")

## Turn `networkx` graph into GNN graph.

In [ ]:
import torch
from torch_geometric.utils.convert import to_networkx, from_networkx

pyg_graph = from_networkx(G)
pyg_graph

## Output GNN graph object.